Kushaan Sharma

# SuperAnimal-TopViewMouse pose estimation on Google Colab (GPU)

GPU version of `python -m behavior_pipeline pose --mode detector`. Runtime -> Change runtime type -> GPU before running.

Measured on the lab's cage videos (2026-09-11): the Faster R-CNN detector finds **nothing on greyscale/CLAHE frames** and works on
colour crops; the pose head scores better on greyscale. So for this route preprocess with `video.grayscale: false` (colour), and
lower the detector threshold (`box_score_thresh`, DLC default 0.6) if mice are missed. On the lab laptop the CPU-only
`pose --mode blob` route (blob boxes + pose head on grey video) is the default and makes Colab optional.

1. Upload the *processed* videos (output of the preprocess stage) to a Drive folder.
2. Run the cells. Results (H5 + labelled video) are written next to each video in Drive.
3. Download the H5 files into `data/pose/raw/` on the lab machine and run `resident` then `convert`.


In [ ]:
!pip -q install "deeplabcut[modelzoo]"
import torch, deeplabcut
print('DLC', deeplabcut.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

VIDEO_DIR = '/content/drive/MyDrive/behavior_pipeline/processed_videos'  # <- edit
DEST_DIR  = '/content/drive/MyDrive/behavior_pipeline/pose_raw'          # <- edit

import glob, os
os.makedirs(DEST_DIR, exist_ok=True)
videos = sorted(glob.glob(f'{VIDEO_DIR}/*.mp4'))
print(len(videos), 'videos'); videos[:5]

In [ ]:
# Mirrors config/pipeline.yaml -> pose.superanimal. video_adapt=True is affordable on a GPU and reduces jitter.
# bbox_threshold only affects plotting in DLC 3.0.1; the real detector filter is detector.model.box_score_thresh,
# set through a customised model config.
import deeplabcut, yaml
from enum import Enum
from pathlib import Path
from deeplabcut.pose_estimation_pytorch.config import PoseConfig

BOX_SCORE_THRESH = 0.3   # 0.6 = DLC default; lower finds more (and more false) boxes

def plain(o):
    if isinstance(o, dict): return {str(k): plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [plain(v) for v in o]
    if isinstance(o, Enum): return plain(o.value)
    if isinstance(o, Path): return str(o)
    return o

cfg = PoseConfig.build_for_superanimal_inference('superanimal_topviewmouse', model_name='hrnet_w32',
                                                 detector_name='fasterrcnn_resnet50_fpn_v2', max_individuals=2, device='cuda')
d = plain(cfg.to_dict()); d['detector']['model']['box_score_thresh'] = BOX_SCORE_THRESH
custom = f'{DEST_DIR}/_superanimal_custom_thr{BOX_SCORE_THRESH}.yaml'
open(custom, 'w').write(yaml.safe_dump(d, sort_keys=False))

deeplabcut.video_inference_superanimal(
    videos=videos,
    superanimal_name='superanimal_topviewmouse',
    model_name='hrnet_w32',
    detector_name='fasterrcnn_resnet50_fpn_v2',
    max_individuals=2,
    video_adapt=True,
    pcutoff=0.1,
    bbox_threshold=0.9,
    batch_size=8,
    dest_folder=DEST_DIR,
    device='cuda',
    create_labeled_video=True,
    customized_model_config=custom,
)
print(sorted(os.listdir(DEST_DIR)))


Check each labelled video: both mice tracked. Back on the lab machine run `python -m behavior_pipeline resident` (decides which
individual is the resident from the pre-entry frames) and then `convert`; `convert --swap <video_name>` overrides that decision.
